<a href="https://colab.research.google.com/github/Ashish265/Deep-learning---Introduction-to-pytorch/blob/main/phase_0_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Check BF16 Support in Code

In [2]:
import torch

print("Is CUDA available: ", torch.cuda.is_available())
print("GPU Device Name: ", torch.cuda.get_device_name(0))
print("Total GPU Count: ", torch.cuda.device_count())
print("Is BF16 format supported: ", torch.cuda.is_bf16_supported())
print("CUDA Compute Capability: ", torch.cuda.get_device_capability(0))

Is CUDA available:  True
GPU Device Name:  Tesla T4
Total GPU Count:  1
Is BF16 format supported:  True
CUDA Compute Capability:  (7, 5)


Inspecting Tokenization in Code

In [3]:
from transformers import AutoTokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")

text = "Finetuning LLMs requires understanding tokenozation"

tokems =  tokenizer.tokenize(text)
ids = tokenizer.encode(text)

print("tokens",tokems)
print("ids",ids)



tokens ['Fin', 'et', 'uning', 'ĠL', 'LM', 's', 'Ġrequires', 'Ġunderstanding', 'Ġtoken', 'oz', 'ation']
ids [9134, 295, 37202, 444, 10994, 82, 7460, 8660, 3950, 9510, 367]


In [11]:
# Check special tokens
print(tokenizer.special_tokens_map)
print(f"BOS id: {tokenizer.bos_token_id}")
print(f"EOS id: {tokenizer.eos_token_id}")
print(f"Vocab size: {tokenizer.vocab_size}")

{'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}
BOS id: None
EOS id: 151643
Vocab size: 151643


 Hardware Audit Script

In [12]:
import subprocess

def hardware_audit():
    if not torch.cuda.is_available():
        print("No CUDA GPU detected. CPU-only mode.")
        return

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\n=== GPU {i}: {props.name} ===")
        print(f"  VRAM:          {props.total_memory / 1e9:.2f} GB")
        print(f"  SM version:    {props.major}.{props.minor}")
        print(f"  BF16 support:  {torch.cuda.is_bf16_supported()}")
        print(f"  Multi-proc SM: {props.multi_processor_count}")

    print(f"\nPyTorch:  {torch.__version__}")
    print(f"CUDA:     {torch.version.cuda}")

hardware_audit()


=== GPU 0: Tesla T4 ===
  VRAM:          15.64 GB
  SM version:    7.5
  BF16 support:  True
  Multi-proc SM: 40

PyTorch:  2.10.0+cu128
CUDA:     12.8


Loading and Inspecting a Model

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [14]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

Count Parameters Per Layer Type

In [15]:
def count_parameters(model):
    total = 0
    layer_counts = {}

    for name, param in model.named_parameters():
        count = param.numel()
        total += count

        # Categorize
        if "q_proj" in name or "k_proj" in name or "v_proj" in name or "o_proj" in name:
            key = "attention"
        elif "gate_proj" in name or "up_proj" in name or "down_proj" in name:
            key = "mlp"
        elif "embed" in name or "lm_head" in name:
            key = "embeddings"
        else:
            key = "other (norm, etc)"

        layer_counts[key] = layer_counts.get(key, 0) + count

    print(f"\nTotal parameters: {total / 1e9:.2f}B")
    print("\nBreakdown:")
    for key, count in sorted(layer_counts.items(), key=lambda x: -x[1]):
        pct = 100 * count / total
        print(f"  {key:30s}: {count/1e6:8.1f}M  ({pct:.1f}%)")

In [16]:
count_parameters(model)


Total parameters: 1.54B

Breakdown:
  mlp                           :   1156.1M  (74.9%)
  embeddings                    :    233.4M  (15.1%)
  attention                     :    154.2M  (10.0%)
  other (norm, etc)             :      0.1M  (0.0%)


Identify LoRA Target Modules

In [17]:
for name, module in model.named_modules():
    if "Linear" in type(module).__name__:
        print(f"{name:60s}  in={module.in_features}, out={module.out_features}")

model.layers.0.self_attn.q_proj                               in=1536, out=1536
model.layers.0.self_attn.k_proj                               in=1536, out=256
model.layers.0.self_attn.v_proj                               in=1536, out=256
model.layers.0.self_attn.o_proj                               in=1536, out=1536
model.layers.0.mlp.gate_proj                                  in=1536, out=8960
model.layers.0.mlp.up_proj                                    in=1536, out=8960
model.layers.0.mlp.down_proj                                  in=8960, out=1536
model.layers.1.self_attn.q_proj                               in=1536, out=1536
model.layers.1.self_attn.k_proj                               in=1536, out=256
model.layers.1.self_attn.v_proj                               in=1536, out=256
model.layers.1.self_attn.o_proj                               in=1536, out=1536
model.layers.1.mlp.gate_proj                                  in=1536, out=8960
model.layers.1.mlp.up_proj                  

In [18]:
inputs = tokenizer("The capital of France is", return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(f"Input shape:  {inputs['input_ids'].shape}")    # (1, 5) — 1 batch, 6 tokens
print(f"Output shape: {logits.shape}")                  # (1, 5, 151936) — logits per token

# Get the predicted next token
next_token_id = logits[0, -1, :].argmax()
print(f"Predicted next token: '{tokenizer.decode(next_token_id)}'")  # Should be "Paris"

Input shape:  torch.Size([1, 5])
Output shape: torch.Size([1, 5, 151936])
Predicted next token: ' Paris'


configure W&B before training:


In [19]:
pip install wandb

In [21]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: msc-ds-2026-04-0078 (msc-ds-2026-04-0078-iiit-hyderabad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Integration with HuggingFaceTrainer / TRL

In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./output",
    report_to="wandb",            # enables W&B logging automatically
    run_name="phase0-baseline",   # name visible in W&B dashboard
    logging_steps=10,             # log every 10 steps
    # ... other args
)

In [24]:
import wandb
wandb.init(
    project="llm-finetuning",     # group all runs under this project
    name="phase0-baseline",
    config={
        "model": "Qwen2.5-1.5B",
        "task": "baseline",
        "dtype": "bf16",
    }
)

What to Log for Every Experiment


In [26]:
experiment_config = {
    # Hardware
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),

    # Model
    "base_model": model,
    "dtype": "bf16",
    "peft_method": "qlora",       # or "lora", "full"

    # Data
    "dataset": "dataset_name",
    "train_samples": 1000,
    "val_samples": 100,
    "max_seq_len": 2048,

    # Training
    "learning_rate": 2e-4,
    "batch_size": 4,
    "grad_accum_steps": 4,
    "effective_batch_size": 16,   # batch_size × grad_accum_steps
    "epochs": 3,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,

    # LoRA (if applicable)
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_target_modules": ["q_proj", "v_proj"],
    "lora_dropout": 0.0,
}

wandb.init(project="llm-finetuning", name="run-01", config=experiment_config)